# PEARL — Exploratory Data Analysis

Quick visual + tabular overview of the Figshare PCOS dataset.
All output is shown inline — there are no on-disk artefacts.

## 0. Setup

Imports only — no torch, no CUDA. Runs in a few seconds on a laptop CPU.

In [ ]:
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = Path("data/raw/figshare")  # infected/, noninfected/
CLASS_MAP = {"noninfected": 0, "infected": 1}

%matplotlib inline


## 1. Inventory + class balance

Count images per class and report the imbalance ratio.

In [ ]:
def inventory(data_dir: Path) -> pd.DataFrame:
    rows = []
    for cls_name, label in CLASS_MAP.items():
        for fp in sorted((data_dir / cls_name).iterdir()):
            if fp.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}:
                rows.append({"path": str(fp), "class_name": cls_name, "label": label})
    return pd.DataFrame(rows)

df = inventory(DATA_DIR)
print(f"Total images: {len(df):,}")
print(df["class_name"].value_counts().to_string())

counts = df["class_name"].value_counts().reindex(["noninfected", "infected"])
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(counts.index, counts.values, color=["tab:gray", "tab:red"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
ax.set_ylabel("Image count")
ax.set_title(f"Class balance (infected / noninfected = {counts['infected']/counts['noninfected']:.2f}x)")
plt.show()


## 2. Image properties

Width, height, and aspect-ratio distribution across the dataset.

In [ ]:
def props(path: str) -> dict:
    try:
        with Image.open(path) as im:
            return {"width": im.width, "height": im.height}
    except Exception:
        return {"width": None, "height": None}

props_df = pd.DataFrame([props(p) for p in df["path"]])
df = pd.concat([df, props_df], axis=1)
df["aspect"] = df["width"] / df["height"]
print("Resolution counts:")
print(df.groupby(["width", "height"]).size().sort_values(ascending=False).head(5).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, label in zip(axes, ["width", "height", "aspect"], ["Width (px)", "Height (px)", "Aspect ratio"]):
    for cls, c in [("noninfected", "tab:gray"), ("infected", "tab:red")]:
        sub = df[df["class_name"] == cls][col].dropna()
        ax.hist(sub, bins=30, alpha=0.6, label=cls, color=c)
    ax.set_xlabel(label); ax.set_ylabel("count")
    ax.legend()
plt.tight_layout(); plt.show()


## 3. Pixel intensity per class

Mean intensity distribution for a 200-per-class sample. Shows whether classes differ in raw brightness.

In [ ]:
rng = np.random.default_rng(42)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (cls_name, label) in zip(axes, CLASS_MAP.items()):
    paths = rng.choice(df[df["label"] == label]["path"].values,
                       size=min(200, (df["label"] == label).sum()), replace=False)
    means = []
    for p in paths:
        im = np.asarray(Image.open(p).convert("L")) / 255.0
        means.append(im.mean())
    ax.hist(means, bins=30, color="tab:red" if label else "tab:gray", edgecolor="k")
    ax.set_title(f"{cls_name} (n={len(means)})")
    ax.set_xlabel("Mean pixel intensity")
    ax.axvline(np.mean(means), color="k", linestyle="--", label=f"mean={np.mean(means):.3f}")
    ax.legend()
axes[0].set_ylabel("count")
plt.tight_layout(); plt.show()


## 4. Sample gallery

Sixteen per class, randomly sampled (seed 0).

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for row, (cls_name, label) in enumerate(CLASS_MAP.items()):
    paths = rng.choice(df[df["label"] == label]["path"].values,
                       size=min(8, (df["label"] == label).sum()), replace=False)
    for col, p in enumerate(paths):
        axes[row, col].imshow(np.asarray(Image.open(p)))
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(cls_name, fontsize=11)
fig.suptitle("Sample gallery (seed=0)", y=1.02)
plt.tight_layout(); plt.show()


## 5. Quality checks

Two silent failure modes:
1. **Exact duplicates** — wasted compute, leakage risk.
2. **Near-blank images** — useless samples that can collapse training.

In [ ]:
# 5a. Exact-byte duplicates via MD5
def md5_of(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

df["md5"] = df["path"].map(md5_of)
unique_per_class = df.drop_duplicates(subset="md5").groupby("class_name").size()
total_per_class  = df.groupby("class_name").size()
fig, ax = plt.subplots(figsize=(7, 3))
for i, cls in enumerate(["noninfected", "infected"]):
    u = unique_per_class.get(cls, 0); t = total_per_class.get(cls, 0)
    ax.bar(cls, t, color="lightgray")
    ax.bar(cls, u, color="tab:red" if cls == "infected" else "dimgray",
           label=f"{u/t:.1%} unique" if cls == "infected" else None)
    ax.text(i, t, f"{u:,} unique / {t:,} total ({u/t:.1%})",
            ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Image count"); ax.set_title("Exact duplicates by class")
ax.set_ylim(0, total_per_class.max() * 1.15)
plt.show()


In [ ]:
# 5b. Near-blank samples (low pixel std). 400-sample estimate is plenty.
sample_for_blank = df.sample(n=min(400, len(df)), random_state=0)
stds = []
for p in sample_for_blank["path"]:
    im = np.asarray(Image.open(p).convert("L")) / 255.0
    stds.append(im.std())
sample_for_blank = sample_for_blank.assign(std=stds)
fig, ax = plt.subplots(figsize=(7, 3))
for cls, c in [("noninfected", "tab:gray"), ("infected", "tab:red")]:
    sub = sample_for_blank[sample_for_blank["class_name"] == cls]["std"]
    ax.hist(sub, bins=30, alpha=0.6, label=cls, color=c)
ax.set_xlabel("Pixel stddev (0 = blank)"); ax.set_ylabel("count")
ax.set_title("Near-blank images (low std)")
ax.legend()
n_blank = int((np.array(stds) < 0.02).sum())
ax.text(0.99, 0.95, f"{n_blank} / {len(stds)} near-blank (std<0.02)",
        ha="right", va="top", transform=ax.transAxes, fontsize=9,
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="gray"))
plt.show()


## 6. Take-aways

One-screen summary for the defense.

In [ ]:
n_unique_total = df.drop_duplicates(subset="md5").shape[0]
takeaways = pd.DataFrame([
    ("Total images", f"{len(df):,}"),
    ("Unique (MD5)", f"{n_unique_total:,}  ({n_unique_total/len(df):.1%})"),
    ("Class ratio (inf:non-inf)", f"{counts['infected']/counts['noninfected']:.2f}:1"),
    ("Modal resolution", f"{df[['width','height']].dropna().astype(int).value_counts().idxmax()[0]}x{df[['width','height']].dropna().astype(int).value_counts().idxmax()[1]}"),
    ("Near-blank (std<0.02)", f"{n_blank} / {len(stds)} sampled"),
], columns=["Metric", "Value"])
takeaways
